# Monosphere プリセット生成

眠っているサブシステムの点火（Fluid/SPH/Metaball/Volumetric/アトラクタ差し替え等）＋アプリ式の交配＋類似度による間引き（greedy max-min）で、新規プリセット48件を生成します。シード固定で再現可能です。

In [ ]:
# Monosphere プリセット生成ノートブック（2026-11-09、Muse が Colab MCP 経由で実行した生成の再現版）
# 使い方: frontier-presets-20261107.json / showcase-presets-20261108.json と
# parameterCatalog.ts（lib/）をこのノートブックと同じ場所に置いて実行してください。
# 生成物: generated.json（アプリ取り込み用）/ gen_report.json
import json
bundle = {"presets": [], "catalog": open("parameterCatalog.ts").read()}
for fn in ("frontier-presets-20261107.json", "showcase-presets-20261108.json"):
    d = json.load(open(fn))
    bundle["presets"] += [{"id": p["id"], "name": p["name"], "config": p["config"]} for p in d["presets"]]
import gzip, base64
open("/tmp/g.b64", "w").write(base64.b64encode(gzip.compress(json.dumps(bundle).encode())).decode())


In [ ]:
bundle = json.loads(gzip.decompress(base64.b64decode(open("/tmp/g.b64").read())))
corpus = bundle["presets"]
catalog_text = bundle["catalog"]

# ---------- catalog ranges ----------
ranges = {}
for m in re.finditer(r"key: '([A-Za-z0-9]+)'[^}]*?min: ([-\d.e+]+), max: ([-\d.e+]+)", catalog_text):
    ranges[m.group(1)] = (float(m.group(2)), float(m.group(3)))
print("catalog ranges parsed:", len(ranges))

# ---------- key universe / reference ----------
union_keys = set()
for p in corpus:
    union_keys |= set(p["config"])
union_keys = sorted(union_keys)
reference = {}
for k in union_keys:
    best = None
    for p in corpus:
        if k in p["config"] and (best is None or len(p["config"]) > len(best[1])):
            best = (p["config"][k], p["config"])
    reference[k] = best[0]
int_keys = set()
for k in union_keys:
    vals = [p["config"][k] for p in corpus if k in p["config"]]
    if vals and all(isinstance(v, (int, float)) and not isinstance(v, bool) and float(v).is_integer() for v in vals):
        int_keys.add(k)

def corpus_range(k):
    vals = [float(p["config"][k]) for p in corpus
            if isinstance(p["config"].get(k), (int, float)) and not isinstance(p["config"].get(k), bool)]
    if not vals:
        return None
    return min(vals), max(vals)

def value_range(k):
    if k in ranges:
        return ranges[k]
    return corpus_range(k)

# ---------- vocabularies ----------
ENUM_ALLOW = ["layer2Source", "layer2Type", "layer3Source", "layer3Type",
              "layer2TemporalProfile", "layer3TemporalProfile",
              "layer2MaterialStyle", "layer3MaterialStyle",
              "layer2GeomMode3D", "layer3GeomMode3D", "layer2SpriteMode", "layer3SpriteMode",
              "layer2ConnectionStyle", "layer3ConnectionStyle",
              "layer2BurstMode", "layer3BurstMode", "layer2BurstWaveform", "layer3BurstWaveform",
              "layer2CollisionMode", "layer3CollisionMode",
              "layer1SdfShape", "layer2SdfShape", "layer3SdfShape", "sdfShape",
              "viewMode", "cameraControlMode", "renderQuality", "postFxStackProfile",
              "synthScale", "audioSourceMode", "interLayerCollisionMode",
              "gpgpuGeomMode", "gpgpuMouseMode", "gpgpuMetaballStyle", "gpgpuExecutionPreference"]
vocab = {}
for k in ENUM_ALLOW:
    vals = sorted({str(p["config"][k]) for p in corpus if k in p["config"]})
    if len(vals) >= 2:
        vocab[k] = vals
vocab["gpgpuAttractorType"] = ["lorenz", "rossler", "thomas"]
vocab["gpgpuVFieldType"] = ["dipole", "saddle", "source", "spiral"]
vocab["gpgpuSdfShape"] = ["box", "capsule", "sphere", "torus"]
vocab["gpgpuEmitShape"] = ["box", "cone", "disc", "ring", "shell", "sphere"]

# ---------- ignition groups ----------
GROUP_DEFS = [
    ("Fluid", "gpgpuFluidEnabled", "gpgpuFluid"),
    ("SPH", "gpgpuSphEnabled", "gpgpuSph"),
    ("Spring", "gpgpuSpringEnabled", "gpgpuSpring"),
    ("Verlet", "gpgpuVerletEnabled", "gpgpuVerlet"),
    ("Elastic", "gpgpuElasticEnabled", "gpgpuElastic"),
    ("Magnetic", "gpgpuMagneticEnabled", "gpgpuMagnetic"),
    ("Metaball", "gpgpuMetaballEnabled", "gpgpuMetaball"),
    ("SDF Collider", "gpgpuSdfEnabled", "gpgpuSdf"),
    ("Tube", "gpgpuTubeEnabled", "gpgpuTube"),
    ("Volumetric", "gpgpuVolumetricEnabled", "gpgpuVolumetric"),
    ("Ambient", "ambientEnabled", "ambient"),
    ("InterLayer Collision", "interLayerCollisionEnabled", "interLayer"),
]
def off_rate(toggle):
    vals = [p["config"].get(toggle) for p in corpus if toggle in p["config"]]
    return 1.0 - (sum(1 for v in vals if v) / len(vals)) if vals else 0.0

groups = []
for title, toggle, stem in GROUP_DEFS:
    if toggle in union_keys and off_rate(toggle) >= 0.9:
        groups.append({"title": title, "toggle": toggle, "stem": stem})
groups.append({"title": "Attractor Swap", "toggle": None, "stem": "gpgpuAttractor"})
groups.append({"title": "VField Swap", "toggle": None, "stem": "gpgpuVField"})
print("ignition groups:", [g["title"] for g in groups])

# missing layer2 combos for coverage targeting
l2_sources = vocab["layer2Source"]; l2_types = vocab["layer2Type"]
present = {(str(p["config"]["layer2Source"]), str(p["config"]["layer2Type"]))
           for p in corpus if "layer2Source" in p["config"] and "layer2Type" in p["config"]}
missing = [(s, t) for s in l2_sources for t in l2_types if (s, t) not in present]
print("missing L2 combos:", len(missing))

# ---------- helpers ----------
def base_config(p):
    return {k: (p["config"][k] if k in p["config"] else reference[k]) for k in union_keys}

def jitter_value(k, v):
    r = value_range(k)
    if r is None:
        return v
    lo, hi = r
    if hi <= lo:
        return v
    nv = lo + py_rng.random() * (hi - lo)
    if k in int_keys:
        nv = int(round(nv))
    return nv

def jitter_key(cfg, k, strength=1.0):
    v = cfg.get(k)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        r = value_range(k)
        if r is None:
            return
        lo, hi = r
        if hi <= lo:
            return
        span = (hi - lo) * 0.3 * strength
        nv = min(hi, max(lo, float(v) + py_rng.uniform(-span, span)))
        cfg[k] = int(round(nv)) if k in int_keys else nv

INHERIT_ONLY = {"layer2MediaLumaMap", "layer3MediaLumaMap", "layer2SourcePositions", "layer3SourcePositions"}

def breed(a, b):
    t = py_rng.uniform(0.25, 0.75)
    out = {}
    for k in union_keys:
        va = a["config"].get(k, reference[k]); vb = b["config"].get(k, reference[k])
        if isinstance(va, (int, float)) and isinstance(vb, (int, float)) and not isinstance(va, bool) and not isinstance(vb, bool):
            nv = float(va) + (float(vb) - float(va)) * t
            out[k] = int(round(nv)) if k in int_keys else nv
        elif isinstance(va, list) and isinstance(vb, list) and k not in INHERIT_ONLY and len(va) == len(vb) and va and all(isinstance(x, (int, float)) for x in va):
            out[k] = [x + (y - x) * t for x, y in zip(va, vb)]
        else:
            out[k] = va if py_rng.random() < 0.5 else vb
    return out

def ignite(cfg, group):
    title = group["title"]
    if title == "Attractor Swap":
        cfg["gpgpuEnabled"] = True
        cfg["gpgpuAttractorEnabled"] = True
        cfg["gpgpuAttractorType"] = py_rng.choice(["rossler", "thomas"])
    elif title == "VField Swap":
        cfg["gpgpuEnabled"] = True
        if "gpgpuVFieldEnabled" in cfg:
            cfg["gpgpuVFieldEnabled"] = True
        cfg["gpgpuVFieldType"] = py_rng.choice(["dipole", "saddle", "source"])
    else:
        cfg[group["toggle"]] = True
        if group["toggle"].startswith("gpgpu"):
            cfg["gpgpuEnabled"] = True
    stem = group["stem"]
    for k in union_keys:
        if k.startswith(stem) and k != group["toggle"]:
            v = cfg.get(k)
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                if py_rng.random() < 0.7:
                    cfg[k] = jitter_value(k, v)
            elif isinstance(v, str) and k in vocab:
                if py_rng.random() < 0.4:
                    cfg[k] = py_rng.choice(vocab[k])
    return title

def mutate(cfg, frac):
    keys = [k for k in union_keys if isinstance(cfg.get(k), (int, float)) and not isinstance(cfg.get(k), bool)]
    for k in py_rng.sample(keys, max(1, int(len(keys) * frac))):
        jitter_key(cfg, k)
    return cfg

def enum_swap(cfg):
    for k in ("layer2Type", "layer3Type", "layer2TemporalProfile", "layer3TemporalProfile",
              "layer2MaterialStyle", "layer3MaterialStyle"):
        if k in vocab and py_rng.random() < 0.5:
            cfg[k] = py_rng.choice(vocab[k])
    if py_rng.random() < 0.6 and missing:
        s, t = py_rng.choice(missing)
        cfg["layer2Source"], cfg["layer2Type"] = s, t
    return cfg

# ---------- candidate generation ----------
gpgpu_bases = [p for p in corpus if p["config"].get("gpgpuEnabled")]
candidates = []
for i in range(N_CANDIDATES):
    r = py_rng.random()
    recipe, applied = None, []
    if r < 0.45:
        recipe = "ignite"
        g = py_rng.choice(groups)
        pool = gpgpu_bases if (g["toggle"] or "").startswith("gpgpu") or g["title"].endswith("Swap") else corpus
        cfg = base_config(py_rng.choice(pool if pool else corpus))
        applied.append(ignite(cfg, g))
        if py_rng.random() < 0.35:
            g2 = py_rng.choice(groups)
            if g2["title"] != applied[0]:
                applied.append(ignite(cfg, g2))
        mutate(cfg, 0.02)
    elif r < 0.75:
        recipe = "breed+ignite"
        a, b = py_rng.sample(corpus, 2)
        cfg = breed(a, b)
        applied.append(ignite(cfg, py_rng.choice(groups)))
        mutate(cfg, 0.02)
    else:
        recipe = "breed/mutate"
        if py_rng.random() < 0.6:
            a, b = py_rng.sample(corpus, 2)
            cfg = breed(a, b)
        else:
            cfg = base_config(py_rng.choice(corpus))
        mutate(cfg, 0.06)
        enum_swap(cfg)
    candidates.append({"config": cfg, "recipe": recipe, "groups": applied})

# ---------- similarity thinning ----------
shared_num = [k for k in union_keys
              if all(isinstance(p["config"].get(k), (int, float)) and not isinstance(p["config"].get(k), bool) for p in corpus)]
print("shared numeric keys:", len(shared_num))
C = np.array([[float(p["config"][k]) for k in shared_num] for p in corpus])
mu, sd = C.mean(axis=0), C.std(axis=0)
keep = sd > 0
def vec(cfg):
    return np.array([float(cfg[k]) for k in shared_num])[keep]
Cz = (C[:, keep] - mu[keep]) / sd[keep]
Cn = Cz / np.where(np.linalg.norm(Cz, axis=1, keepdims=True) == 0, 1, np.linalg.norm(Cz, axis=1, keepdims=True))
X = np.array([vec(c["config"]) for c in candidates])
Xz = (X - mu[keep]) / sd[keep]
Xn = Xz / np.where(np.linalg.norm(Xz, axis=1, keepdims=True) == 0, 1, np.linalg.norm(Xz, axis=1, keepdims=True))
sim_corpus = Xn @ Cn.T
max_to_corpus = sim_corpus.max(axis=1)
nearest_corpus = sim_corpus.argmax(axis=1)

selected, sel_sims = [], []
type_count = Counter()
ignition_selected = 0
remaining = set(range(len(candidates)))
cur_max = max_to_corpus.copy()
while len(selected) < N_SELECT and remaining:
    best, best_score = None, -2
    for i in remaining:
        c = candidates[i]
        t = str(c["config"].get("layer2Type"))
        if type_count[t] >= 3:
            continue
        score = -cur_max[i]
        if score > best_score:
            best, best_score = i, score
    if best is None:
        break
    selected.append(best); remaining.discard(best)
    type_count[str(candidates[best]["config"].get("layer2Type"))] += 1
    if candidates[best]["groups"]:
        ignition_selected += 1
    sim_new = Xn @ Xn[best]
    cur_max = np.maximum(cur_max, sim_new)
print("selected:", len(selected), "with ignition:", ignition_selected)

# ---------- assemble + validate ----------
now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
def titleize(s):
    return " ".join(w.capitalize() for w in str(s).replace("_", " ").split())
used_names = {p["name"] for p in corpus}
presets_out, meta = [], []
for n, i in enumerate(selected, 1):
    c = candidates[i]
    cfg = c["config"]
    gtitle = " + ".join(c["groups"]) if c["groups"] else "Cross"
    name = f"Ignition {n:02d} — {gtitle} {titleize(cfg.get('layer2Type', ''))}"
    if name in used_names:
        name += " II"
    used_names.add(name)
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    pid = f"{slug}-20261109"
    presets_out.append({"id": pid, "name": name, "config": cfg, "createdAt": now, "updatedAt": now})
    meta.append({"name": name, "id": pid, "recipe": c["recipe"], "groups": c["groups"],
                 "layer2Type": cfg.get("layer2Type"), "layer2Source": cfg.get("layer2Source"),
                 "max_sim_to_corpus": round(float(max_to_corpus[i]), 4),
                 "nearest_corpus": corpus[int(nearest_corpus[i])]["name"]})

errors = []
ref_types = {k: type(reference[k]).__name__ for k in union_keys}
for p in presets_out:
    if set(p["config"]) != set(union_keys):
        errors.append(f"{p['name']}: key set mismatch")
    for k, v in p["config"].items():
        rt = ref_types[k]
        if rt == "bool" and not isinstance(v, bool):
            errors.append(f"{p['name']}:{k} not bool")
        elif rt in ("int", "float") and (not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(float(v))):
            errors.append(f"{p['name']}:{k} bad number")
        elif rt == "str" and not isinstance(v, str):
            errors.append(f"{p['name']}:{k} not str")
        if k in vocab and isinstance(v, str) and v not in vocab[k]:
            errors.append(f"{p['name']}:{k} illegal enum {v}")
print("validation errors:", len(errors), errors[:5])

envelope = {"version": 1, "activePresetId": presets_out[0]["id"],
            "currentConfig": presets_out[0]["config"], "exportedAt": now,
            "presetBlendDuration": 1.4, "presetSequence": [],
            "presets": presets_out, "sequenceLoopEnabled": True}
with open("/tmp/generated.json", "w") as fh:
    json.dump(envelope, fh, ensure_ascii=False)
blob = base64.b64encode(gzip.compress(json.dumps(envelope).encode())).decode()
open("/tmp/generated.json.gz.b64", "w").write(blob)
report = {"seed": SEED, "candidates": N_CANDIDATES, "selected": len(presets_out),
          "groups_available": [g["title"] for g in groups],
          "group_counts": dict(Counter(g for m in meta for g in m["groups"])),
          "recipe_counts": dict(Counter(m["recipe"] for m in meta)),
          "validation_errors": errors, "presets": meta}
with open("/tmp/gen_report.json", "w") as fh:
    json.dump(report, fh, ensure_ascii=False)
print("generated.json.gz.b64 chars:", len(blob))
print("group counts:", report["group_counts"])
print("DONE")
